# Audio Segmentation Pipeline (PD vs HC)

This notebook:
1. Reads `final_selected.xlsx` and separates participants into **PD** and **HC** cohorts.
2. Maps each FLAC file to its cohort using the `recordId` prefix in the filename.
3. Filters files shorter than 10 s (cannot yield all three segments).
4. Splits qualifying files into three fixed-length segments and saves them under `segments/PD/` and `segments/HC/`.

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path

In [ ]:
print(f"pandas     version : {pd.__version__}")
print(f"soundfile  version : {sf.__version__}")
print(f"Supported formats  : {list(sf.available_formats().keys())[:10]} ...")

## 2. Load Cohort Labels from `final_selected.xlsx`

Read the metadata file and build two sets of `recordId` values — one for **PD** and one for **HC**.  
Rows with `cohort == "Unknown"` are ignored.

In [ ]:
EXCEL_PATH = Path("final_selected.xlsx")

meta = pd.read_excel(EXCEL_PATH)

# Keep only PD and HC rows
meta = meta[meta["cohort"].isin(["PD", "HC"])].copy()

pd_record_ids = set(meta.loc[meta["cohort"] == "PD", "recordId"].astype(str))
hc_record_ids = set(meta.loc[meta["cohort"] == "HC", "recordId"].astype(str))

print(f"Unique PD recordIds : {len(pd_record_ids)}")
print(f"Unique HC recordIds : {len(hc_record_ids)}")

## 3. Scan FLAC Files and Assign to Cohort

Each FLAC filename has the format `{recordId}_{fileId}.flac`.  
Files whose `recordId` belongs to PD are assigned to the **PD** group; likewise for **HC**.  
Files shorter than 10 s are discarded.

In [ ]:
AUDIO_DIR      = Path("mpower_voice_data_flac-20260127T044147Z-3-012/mpower_voice_data_flac")
MIN_DURATION_S = 10.0   # seconds

# cohort_files["PD"] / ["HC"] -> list of valid Paths
cohort_files    = {"PD": [], "HC": []}
discarded_files = []   # (Path, duration_s, reason)
unmatched_files = []   # files not in PD or HC

all_flac_files = sorted(AUDIO_DIR.glob("*.flac"))

for fpath in all_flac_files:
    # Extract recordId = everything before the first '_'
    record_id = fpath.stem.split("_")[0]

    if record_id in pd_record_ids:
        cohort = "PD"
    elif record_id in hc_record_ids:
        cohort = "HC"
    else:
        unmatched_files.append(fpath)
        continue

    try:
        info = sf.info(fpath)
        if info.duration >= MIN_DURATION_S:
            cohort_files[cohort].append(fpath)
        else:
            discarded_files.append((fpath, info.duration, "too short"))
    except Exception as exc:
        discarded_files.append((fpath, -1, str(exc)))

print(f"Total FLAC files found  : {len(all_flac_files)}")
print(f"Unmatched (not PD/HC)   : {len(unmatched_files)}")
print(f"Discarded (< 10 s)      : {len(discarded_files)}")
print(f"Valid PD files          : {len(cohort_files['PD'])}")
print(f"Valid HC files          : {len(cohort_files['HC'])}")

## 4. Define Segments and Create Output Folders

Segment definitions (in seconds):

| Segment | Start | End  |
|---------|-------|------|
| Early   | 0 s   | 3 s  |
| Middle  | 3 s   | 7 s  |
| Late    | 7 s   | 10 s |

Output structure:
```
segments/
  PD/
    early/
    middle/
    late/
  HC/
    early/
    middle/
    late/
```

In [ ]:
SEGMENTS = {
    "early":  (0,  3),
    "middle": (3,  7),
    "late":   (7, 10),
}

OUTPUT_BASE = Path("segments")
COHORTS     = ["PD", "HC"]

for cohort in COHORTS:
    for seg_name in SEGMENTS:
        (OUTPUT_BASE / cohort / seg_name).mkdir(parents=True, exist_ok=True)

print("Output folders created:")
for cohort in COHORTS:
    for seg_name in SEGMENTS:
        print(f"  {OUTPUT_BASE / cohort / seg_name}")

## 5. Segment and Save — PD & HC

Each valid file is sliced into Early / Middle / Late and exported as a FLAC file.  
Naming convention: `{original_stem}_{segment_label}.flac`  
e.g. `ff08e01d-3971-4de9-a2c1-584ecc681db0_6000123_early.flac`

In [ ]:
saved_counts = {"PD": 0, "HC": 0}
failed_files = []

for cohort in COHORTS:
    for fpath in cohort_files[cohort]:
        try:
            data, samplerate = sf.read(str(fpath), always_2d=True)
        except Exception as exc:
            failed_files.append((fpath.name, cohort, str(exc)))
            continue

        for seg_name, (start_s, end_s) in SEGMENTS.items():
            start_sample = int(start_s * samplerate)
            end_sample   = int(end_s   * samplerate)
            segment_data = data[start_sample:end_sample]

            out_filename = f"{fpath.stem}_{seg_name}.flac"
            out_path     = OUTPUT_BASE / cohort / seg_name / out_filename
            sf.write(str(out_path), segment_data, samplerate)

        saved_counts[cohort] += 1

print("Segments saved successfully:")
for cohort in COHORTS:
    print(f"  {cohort}: {saved_counts[cohort]} files  ->  {OUTPUT_BASE / cohort}/")
print(f"\nFiles skipped (unreadable): {len(failed_files)}")
if failed_files:
    print("\nSkipped files:")
    for name, cohort, reason in failed_files:
        print(f"  [{cohort}] {name}  ->  {reason}")
print(f"\nOutput root: {OUTPUT_BASE.resolve()}")

# Audio Segmentation Pipeline (PD vs HC)

This notebook:
1. Reads `final_selected.xlsx` and separates participants into **PD** and **HC** cohorts.
2. Maps each FLAC file to its cohort using the `recordId` prefix in the filename.
3. Filters files shorter than 10 s (cannot yield all three segments).
4. Splits qualifying files into three fixed-length segments and saves them under `segments/PD/` and `segments/HC/`.


## 1. Import Required Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path


In [2]:
print(f"pandas     version : {pd.__version__}")
print(f"soundfile  version : {sf.__version__}")
print(f"Supported formats  : {list(sf.available_formats().keys())[:10]} ...")


pandas     version : 3.0.1
soundfile  version : 0.13.1
Supported formats  : ['AIFF', 'AU', 'AVR', 'CAF', 'FLAC', 'HTK', 'SVX', 'MAT4', 'MAT5', 'MPC2K'] ...


## 2. Load Cohort Labels from `final_selected.xlsx`

Read the metadata file and build two sets of `recordId` values — one for **PD** and one for **HC**.  
Rows with `cohort == "Unknown"` are ignored.


In [3]:
EXCEL_PATH = Path("final_selected.xlsx")

meta = pd.read_excel(EXCEL_PATH)

# Keep only PD and HC rows
meta = meta[meta["cohort"].isin(["PD", "HC"])].copy()

pd_record_ids = set(meta.loc[meta["cohort"] == "PD", "recordId"].astype(str))
hc_record_ids = set(meta.loc[meta["cohort"] == "HC", "recordId"].astype(str))

print(f"Unique PD recordIds : {len(pd_record_ids)}")
print(f"Unique HC recordIds : {len(hc_record_ids)}")


Unique PD recordIds : 9929
Unique HC recordIds : 23086


## 3. Scan FLAC Files and Assign to Cohort

Each FLAC filename has the format `{recordId}_{fileId}.flac`.  
Files whose `recordId` belongs to PD are assigned to the **PD** group; likewise for **HC**.  
Files shorter than 10 s are discarded.


In [4]:
AUDIO_DIR     = Path("mpower_voice_data_flac-20260127T044147Z-3-012/mpower_voice_data_flac")
MIN_DURATION_S = 10.0   # seconds

# cohort_files["PD"] / ["HC"] → list of valid Paths
cohort_files    = {"PD": [], "HC": []}
discarded_files = []   # (Path, duration_s, reason)
unmatched_files = []   # files not in PD or HC

all_flac_files = sorted(AUDIO_DIR.glob("*.flac"))

for fpath in all_flac_files:
    # Extract recordId = everything before the first '_'
    record_id = fpath.stem.split("_")[0]

    if record_id in pd_record_ids:
        cohort = "PD"
    elif record_id in hc_record_ids:
        cohort = "HC"
    else:
        unmatched_files.append(fpath)
        continue

    try:
        info = sf.info(fpath)
        if info.duration >= MIN_DURATION_S:
            cohort_files[cohort].append(fpath)
        else:
            discarded_files.append((fpath, info.duration, "too short"))
    except Exception as exc:
        discarded_files.append((fpath, -1, str(exc)))

print(f"Total FLAC files found  : {len(all_flac_files)}")
print(f"Unmatched (not PD/HC)   : {len(unmatched_files)}")
print(f"Discarded (< 10 s)      : {len(discarded_files)}")
print(f"Valid PD files          : {len(cohort_files['PD'])}")
print(f"Valid HC files          : {len(cohort_files['HC'])}")


Total FLAC files found  : 2433
Unmatched (not PD/HC)   : 1176
Discarded (< 10 s)      : 8
Valid PD files          : 338
Valid HC files          : 911


## 4. Define Segments and Create Output Folders

Segment definitions (in seconds):

| Segment | Start | End  |
|---------|-------|------|
| Early   | 0 s   | 3 s  |
| Middle  | 3 s   | 7 s  |
| Late    | 7 s   | 10 s |

Output structure:
```
segments/
  PD/
    early/
    middle/
    late/
  HC/
    early/
    middle/
    late/
```


In [5]:
SEGMENTS = {
    "early":  (0,  3),
    "middle": (3,  7),
    "late":   (7, 10),
}

OUTPUT_BASE = Path("segments")
COHORTS     = ["PD", "HC"]

for cohort in COHORTS:
    for seg_name in SEGMENTS:
        (OUTPUT_BASE / cohort / seg_name).mkdir(parents=True, exist_ok=True)

print("Output folders created:")
for cohort in COHORTS:
    for seg_name in SEGMENTS:
        print(f"  {OUTPUT_BASE / cohort / seg_name}")


Output folders created:
  segments/PD/early
  segments/PD/middle
  segments/PD/late
  segments/HC/early
  segments/HC/middle
  segments/HC/late


## 5. Segment and Save — PD & HC

Each valid file is sliced into Early / Middle / Late and exported as a FLAC file.  
Naming convention: `{original_stem}_{segment_label}.flac`  
e.g. `ff08e01d-3971-4de9-a2c1-584ecc681db0_6000123_early.flac`


In [6]:
saved_counts = {"PD": 0, "HC": 0}
failed_files = []

for cohort in COHORTS:
    for fpath in cohort_files[cohort]:
        try:
            data, samplerate = sf.read(str(fpath), always_2d=True)
        except Exception as exc:
            failed_files.append((fpath.name, cohort, str(exc)))
            continue

        for seg_name, (start_s, end_s) in SEGMENTS.items():
            start_sample = int(start_s * samplerate)
            end_sample   = int(end_s   * samplerate)
            segment_data = data[start_sample:end_sample]

            out_filename = f"{fpath.stem}_{seg_name}.flac"
            out_path     = OUTPUT_BASE / cohort / seg_name / out_filename
            sf.write(str(out_path), segment_data, samplerate)

        saved_counts[cohort] += 1

print("Segments saved successfully:")
for cohort in COHORTS:
    print(f"  {cohort}: {saved_counts[cohort]} files  →  {OUTPUT_BASE / cohort}/")
print(f"\nFiles skipped (unreadable): {len(failed_files)}")
if failed_files:
    print("\nSkipped files:")
    for name, cohort, reason in failed_files:
        print(f"  [{cohort}] {name}  ->  {reason}")
print(f"\nOutput root: {OUTPUT_BASE.resolve()}")


Segments saved successfully:
  PD: 338 files  →  segments/PD/
  HC: 909 files  →  segments/HC/

Files skipped (unreadable): 2

Skipped files:
  [HC] 1f3e285b-3677-4e2c-9bb1-88e1866f974f_6053206.flac  ->  Internal psf_fseek() failed.
  [HC] 3f045d84-fe8c-467f-ad78-4a12951c9c29_6054644.flac  ->  Internal psf_fseek() failed.

Output root: /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/segments
